# L30 · MLOps：让 AI 系统稳定运转

**学习目标**
- 理解 MLOps：模型从「训练」到「稳定服务」的全生命周期管理
- 掌握关键机制：版本管理、金丝雀发布、自动回滚、监控
- 亲手模拟一条「部署流水线」，体验安全上线与回滚

**前置依赖**：L25（评测）、L27（可观测）、L29（成本）  
**预计时长**：50 分钟  
**技术栈**：纯 Python（离线流水线模拟）

---\n
## 概念讲解：MLOps = 让 AI 像工厂产品一样可靠

训练出一个好模型只是 5% 的工作。剩下 95% 是：**怎么安全地把它放到线上、持续监控、出问题秒回滚**。

核心动作：
- **版本管理**：每个模型有编号，可回溯
- **金丝雀发布（Canary）**：先放 10% 流量给新模型试水
- **自动回滚**：监控到错误率飙升，立刻切回旧版

本课我们模拟这条流水线，看一次「安全上线 + 异常回滚」全过程。

## 第一步：模型注册表（版本管理）

In [ ]:
models = {
    "v1.0": {"acc": 0.90, "status": "stable"},
    "v1.1": {"acc": 0.93, "status": "candidate"},  # 新候选，待发布
}
current = "v1.0"
print("模型注册表：", models)
print("当前线上版本：", current)

## 第二步：金丝雀发布 + 监控 + 自动回滚逻辑

In [ ]:
def canary_release(candidate, baseline_acc, canary_acc):
    """模拟：先放小流量，看指标；掉太多就回滚"""
    logs = []
    logs.append(f"  🚀 开始金丝雀：10% 流量切到 {candidate}")
    # 模拟监控到的准确率
    logs.append(f"  📡 监控：{candidate} 金丝雀准确率 = {canary_acc:.2f}")
    if canary_acc < baseline_acc - 0.05:   # 掉超过 5 个点
        logs.append(f"  🔄 触发回滚：准确率骤降，切回稳定版 {current}")
        return current, logs
    else:
        logs.append(f"  ✅ 金丝雀通过，全量发布 {candidate}")
        return candidate, logs

new_ver, log = canary_release("v1.1", models[current]["acc"], 0.94)
for line in log: print(line)

# 🎯 AHA 顿悟单元格：你的「AI 部署流水线」实况

运行下面代码。你会看到一条完整流水线**自动演示两种结局**：
① 新模型表现好 → 顺利全量发布；② 新模型翻车（准确率低）→ **自动回滚**保命。
改 `canary_acc` 体验两种分支。

> 你刚模拟的，就是 Google/Meta 每天在做的「安全发布」。没有这套机制，一次坏模型上线能酿成大事故。

In [ ]:
# ===== 运行我！看流水线两种结局 =====
print("  ⚙️  MLOps 部署流水线 · 实况模拟\n")
for scenario, canary_acc in [("场景A 新模型更优", 0.96), ("场景B 新模型翻车", 0.82)]:
    print(f"  ── {scenario}（金丝雀准确率 {canary_acc}）──")
    nv, lg = canary_release("v1.1", models[current]["acc"], canary_acc)
    for l in lg: print("  " + l)
    print()
print("  ✨ 你拥有了『让 AI 系统永不崩』的工程护盾！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：金丝雀/回滚心智；MLOps 是「研发+运维」交叉学科，需强调它独立于模型算法本身。  
**易错点**：回滚判定阈值（本方案 -0.05）应说明是示例；真实用置信区间/统计检验。  
**AHA 机制**：两种结局对比，强「系统可靠性」工程震撼。  
**衔接**：阶段六（L31+ 后训练，训练出的模型走这条流水线发布）；L37-40 综合项目。  
**依赖**：纯 Python 标准库。  
**SOTA 实践**：对标 MLflow（版本）、Kubernetes（编排）、Prometheus（监控）、Argo Rollouts（金丝雀）。  
**阶段五收尾**：此课是工程化约束 AI 阶段终点，AHA 要体现「系统级稳健」。

# 📚 作业 / 下一步

1. 把阈值 `-0.05` 改成 `-0.02`，看回滚更敏感。
2. 加一个「监控延迟」维度参与回滚决策。
3. 进入 **阶段六 · 后训练与强化学习**：L31 后训练总览 —— 揭开 SFT/DPO/PPO 的神秘面纱。